# Land availability

All heavy preprocessing is handled by `model/land_processing.py`, keeping these cells
short and easy to re-run on a new workstation.


In [ ]:
from pathlib import Path
import sys
import importlib

import pandas as pd
import plotly.express as px

def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start] + list(start.parents):
        if (p / 'model').is_dir():
            return p
    raise RuntimeError("Could not locate repo root containing 'model/'")

NOTEBOOK_DIR = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
PROJECT_ROOT = find_repo_root(NOTEBOOK_DIR)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from model import land_processing as lp
lp = importlib.reload(lp)

pd.options.display.float_format = '{:,.2f}'.format


In [ ]:
DATA_DIR = PROJECT_ROOT / "data"
LAND_COVER_FILE = DATA_DIR / "MCD12C1.A2022001.061.2023244164746.hdf"
OUTPUT_CSV = DATA_DIR / "20251222_land_max_capacity.csv"

config = lp.LandAvailabilityConfig(
    land_cover_path=LAND_COVER_FILE,
    output_csv=OUTPUT_CSV,
)
config


In [ ]:
land_df = lp.write_land_availability_table(config)
land_df.head()

In [ ]:
summary = land_df[["availability", "max_capacity", "area"]].describe(percentiles=[0.5, 0.9, 0.99])
summary

total_capacity_mw = land_df["max_capacity"].sum()
print(f"Total installable capacity: {total_capacity_mw:,.0f} MW")

In [ ]:
top_sites = land_df.nlargest(10, "max_capacity")[["latitude", "longitude", "availability", "max_capacity"]]
top_sites

In [ ]:
def _cell_id(lat, lon, decimals=2):
    return f"{lat:.{decimals}f}_{lon:.{decimals}f}"


def _build_cell_geojson(df, lat_col="latitude", lon_col="longitude", id_col="cell_id", cell_size_deg=1.0):
    half = cell_size_deg / 2.0
    features = []
    for lat, lon, cell_id in zip(df[lat_col], df[lon_col], df[id_col]):
        lat0 = float(lat)
        lon0 = float(lon)
        polygon = [
            [lon0 - half, lat0 - half],
            [lon0 - half, lat0 + half],
            [lon0 + half, lat0 + half],
            [lon0 + half, lat0 - half],
            [lon0 - half, lat0 - half],
        ]
        features.append(
            {
                "type": "Feature",
                "id": cell_id,
                "properties": {"latitude": lat0, "longitude": lon0},
                "geometry": {"type": "Polygon", "coordinates": [polygon]},
            }
        )
    return {"type": "FeatureCollection", "features": features}


cell_size_deg = getattr(config, "coarse_degree", 1.0)
plot_df = land_df.loc[land_df["max_capacity"] > 0, ["latitude", "longitude", "max_capacity", "availability"]].copy()
plot_df["cell_id"] = [_cell_id(lat, lon) for lat, lon in zip(plot_df["latitude"], plot_df["longitude"])]

cell_geojson = _build_cell_geojson(plot_df, cell_size_deg=cell_size_deg)

fig = px.choropleth_map(
    plot_df,
    geojson=cell_geojson,
    locations="cell_id",
    featureidkey="id",
    color="max_capacity",
    map_style="carto-positron",
    color_continuous_scale="Viridis",
    opacity=0.85,
    labels={"max_capacity": "Max Capacity (MW)", "availability": "Availability"},
    hover_data={"latitude": True, "longitude": True, "max_capacity": True, "availability": True},
    center=dict(lat=0, lon=0),
    zoom=0.5,
)
fig.update_traces(marker_line_width=0.2, marker_line_color="rgba(255, 255, 255, 0.3)")
fig.update_layout(margin=dict(l=0, r=0, t=0, b=0))
fig.show()

In [ ]:
hist = px.histogram(land_df, x="max_capacity", nbins=60, title="Max capacity distribution")
hist.update_layout(xaxis_title="max_capacity (MW)")
hist.show()